# 07-1 — Observability with Tempo / OpenTelemetry

Demonstrates distributed tracing for the agent framework via **OpenTelemetry + Grafana Tempo**.

Every agent run, tool call, and LLM request is captured as a span and exported  
to a Tempo backend over OTLP (HTTP).

**Stack**:
- `opentelemetry-sdk` — spans and exporters
- `opentelemetry-exporter-otlp-proto-http` — OTLP/HTTP to Grafana Tempo
- Grafana at `http://localhost:3001` (started via `docker compose -f docker/docker-compose.yml`)

**Prerequisites**:
- `docker compose -f docker/docker-compose.yml up -d tempo grafana` (or your OTLP-compatible backend)
- `OPENAI_API_KEY` set

In [ ]:
import os
import asyncio

OTLP_ENDPOINT = os.environ.get("OTEL_EXPORTER_OTLP_TRACES_ENDPOINT", "http://localhost:4318")
from ravi.configs.settings import settings

CHAT_MODEL = settings.CHAT_MODEL
API_KEYS = {
    "openai":     settings.OPENAI_API_KEY,
    "anthropic":  settings.ANTHROPIC_API_KEY,
    "google":     settings.GEMINI_API_KEY,
    "groq":       settings.GROQ_API_KEY,
    "openrouter": settings.OPENROUTER_API_KEY,
}
print(f"OTLP endpoint: {OTLP_ENDPOINT}")
print(f"CHAT_MODEL: {CHAT_MODEL}")

OTLP endpoint: http://localhost:4318
CHAT_MODEL: openai/gpt-5.4-mini


: 

In [ ]:
from ravi.core.agent_catalog import AgentCatalog
from ravi.core.agents.react_agent import ReActAgent
from ravi.core.tools.builtin_tools import CalculatorTool, GetCurrentTimeTool
from ravi.integrations.llm.factory import create_model_client
from ravi.core.memory.unbounded_memory import UnboundedMemory
from ravi.core.context.implementations import UnboundedContext
from ravi.console import Console
from ravi.shared.observability import configure_opentelemetry

## Configure OpenTelemetry

In [3]:
configure_opentelemetry(
    service_name='agent-framework-tempo-demo',
    otlp_trace_endpoint=OTLP_ENDPOINT,
)
print('OpenTelemetry configured ✅')

Configuring OpenTelemetry
Using OTLP HTTP trace exporter → http://localhost:4318/v1/traces
Metrics export is disabled (no OTLP metric endpoint and console export disabled)
OpenTelemetry configured ✅


## Run a traced agent

After this cell completes, open **Grafana → Explore → Tempo** and filter  
by service name `agent-framework-tempo-demo` to see the trace.

In [ ]:
async def run():
    catalog = AgentCatalog()
    catalog.register_model("primary", create_model_client(CHAT_MODEL, api_keys=API_KEYS))
    catalog.register_memory("default", UnboundedMemory())
    catalog.register_context("default", UnboundedContext())
    catalog.register_tool(CalculatorTool())
    catalog.register_tool(GetCurrentTimeTool())

    agent = ReActAgent(
        name="TempoDemoBot",
        description="A helpful assistant for demonstrating tracing.",
        catalog=catalog,
        max_iterations=5,
        verbose=True,
    )

    print(f"Tracing model: {CHAT_MODEL}")
    result = await Console(agent).run("Calculate 123 * 456 and tell me the current time.")

    # Give the OTLP exporter time to flush
    await asyncio.sleep(2)
    print("\nTraces flushed — check Grafana at http://localhost:3001")

await run()

You → Calculate 123 * 456 and tell me the current time.

╭───────────────────────────────────────────────── TempoDemoBot ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  The result of (123 \times 456) is (56,088).                                                                    │
│                                                                                                                 │
│  The current time in UTC is 19:12 on March 15, 2026.                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

completed · 2 steps · 2 tool calls · 511 tokens · 3.2s


Traces flushed — check Grafana at http://localhost:3001


---
## Trace context propagation through EventBus (Sprint 8)

When a service publishes an event, the current OTel span context
(W3C  / ) is stamped into .
The consuming service extracts it and links its processing span back to the
producer — giving end-to-end distributed traces across service boundaries.

**** — new  field  
**** — injects current span  
**** — extracts span and starts linked consumer span

In [ ]:
from ravi.shared.events.envelope import EventEnvelope
from opentelemetry.propagate import inject, extract

# --- Publishing side ---
# EventBus.publish() does this automatically. Here we show it explicitly:
carrier: dict[str, str] = {}
inject(carrier)   # stamps the active OTel span into carrier

event = EventEnvelope(
    event_type="workflow.run_started",
    payload={"run_id": "abc-123", "thread_id": "t-456"},
    trace_context=carrier,   # ← new field (Sprint 8)
)

print(f"event.event_type   : {event.event_type}")
print(f"event.trace_context: {event.trace_context}")
print(f"  (empty when no active span — run inside a traced agent to see traceparent)")

# --- Consuming side ---
# EventBus.subscribe() does this automatically. Shown explicitly:
parent_ctx = extract(event.trace_context)
from opentelemetry import trace
tracer = trace.get_tracer("demo")
with tracer.start_as_current_span(
    f"consume:{event.event_type}",
    context=parent_ctx,
    kind=trace.SpanKind.CONSUMER,
) as span:
    print(f"Consumer span: {span.get_span_context().span_id:#018x}")
    print("Span is linked to the publisher trace — visible as one trace in Tempo")